# PFE ML Data Pipeline Workbook

This notebook runs the project data pipeline from Google Colab.

**Recommended runtime:** Python 3, CPU. Use High RAM only for large exports/builds or if memory errors appear.

**Persistent data folder:** `/content/drive/MyDrive/PFE ML Data/pfe_data`

**Temporary code folder:** `/content/pfein`


In [2]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
DRIVE_ROOT = '/content/drive/MyDrive/PFE ML Data/pfe_data'
REPO_URL = 'https://github.com/zribi1/pfein.git'
BRANCH = 'data-extraction'
REPO_DIR = '/content/pfein'
BACKEND_DIR = '/content/pfein/back_end'

!mkdir -p "$DRIVE_ROOT"
!ls -lah "$DRIVE_ROOT"


total 16K
drwx------ 6 root root 4.0K May  8 22:35 data-lake
drwx------ 2 root root 4.0K May  8 22:35 ml-artifacts
drwx------ 2 root root 4.0K May  9 01:01 reports
drwx------ 5 root root 4.0K May  8 22:35 source-archives


## 1. Clone Or Refresh Code

Run this cell at the start of a fresh Colab session. It clones the repo if missing, otherwise it pulls the latest code.

In [4]:
import os

if not os.path.exists(REPO_DIR):
    %cd /content
    !git clone --branch "$BRANCH" --single-branch "$REPO_URL" "$REPO_DIR"
else:
    %cd $REPO_DIR
    !git pull

%cd $BACKEND_DIR


/content
Cloning into '/content/pfein'...
remote: Enumerating objects: 308, done.
remote: Counting objects: 100% (308/308), done.
remote: Compressing objects: 100% (238/238), done.
remote: Total 308 (delta 80), reused 279 (delta 51), pack-reused 0 (from 0)
Receiving objects: 100% (308/308), 457.26 KiB | 15.24 MiB/s, done.
Resolving deltas: 100% (80/80), done.
/content/pfein/back_end


## 2. Install Colab Dependencies

Use `collabs/requirements-colab.txt`. Do not install the full backend requirements unless you want to run the API inside Colab.

In [5]:
%cd $BACKEND_DIR
!pip install -q -r collabs/requirements-colab.txt


/content/pfein/back_end
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.2/20.2 MB 96.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.9/39.9 MB 57.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 227.3/227.3 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 67.7 MB/s eta 0:00:00


## 3. Quick Environment Check

In [3]:
!python --version
!df -h /content
!ls -lah "$DRIVE_ROOT"
!find "$DRIVE_ROOT" -maxdepth 2 -type d | sort | head -80


Python 3.12.13
Filesystem      Size  Used Avail Use% Mounted on
overlay         226G   24G  203G  11% /
total 16K
drwx------ 6 root root 4.0K May  8 22:35 data-lake
drwx------ 2 root root 4.0K May  8 22:35 ml-artifacts
drwx------ 2 root root 4.0K May  9 01:01 reports
drwx------ 5 root root 4.0K May  8 22:35 source-archives
/content/drive/MyDrive/PFE ML Data/pfe_data
/content/drive/MyDrive/PFE ML Data/pfe_data/data-lake
/content/drive/MyDrive/PFE ML Data/pfe_data/data-lake/clean
/content/drive/MyDrive/PFE ML Data/pfe_data/data-lake/features
/content/drive/MyDrive/PFE ML Data/pfe_data/data-lake/raw
/content/drive/MyDrive/PFE ML Data/pfe_data/data-lake/tmp
/content/drive/MyDrive/PFE ML Data/pfe_data/ml-artifacts
/content/drive/MyDrive/PFE ML Data/pfe_data/reports
/content/drive/MyDrive/PFE ML Data/pfe_data/source-archives
/content/drive/MyDrive/PFE ML Data/pfe_data/source-archives/bodacc
/content/drive/MyDrive/PFE ML Data/pfe_data/source-archives/financials
/content/drive/MyDrive/PFE ML D

## 4. Public Data Pipeline: INSEE + Financials

Run this when starting from zero or when refreshing INSEE and financial data. It skips INPI and BODACC.

In [ ]:
%cd $BACKEND_DIR
!python collabs/full_pipeline.py \
  --drive-root "$DRIVE_ROOT" \
  --install-deps \
  --no-inpi \
  --no-bodacc \
  --start-year 2017 \
  --end-year 2025 \
  --max-companies 100000


## 5. Rebuild From Existing Downloads Only

Use this after code fixes or after adding a source. It does not redownload INSEE or financial files.

In [ ]:
%cd $BACKEND_DIR
!python collabs/build_ml_data.py \
  --drive-root "$DRIVE_ROOT" \
  --start-year 2017 \
  --end-year 2025 \
  --max-companies 100000


## 6. Audit Data Lake

Run after each major change. The timestamped report is written to Google Drive.

In [7]:
from datetime import datetime

RUN_TAG = datetime.utcnow().strftime('%Y_%m_%d_%H%M')
AUDIT_MD = f'{DRIVE_ROOT}/reports/data_lake_audit_{RUN_TAG}.md'
AUDIT_JSON = f'{DRIVE_ROOT}/reports/data_lake_audit_{RUN_TAG}.json'
print(AUDIT_MD)


/content/drive/MyDrive/PFE ML Data/pfe_data/reports/data_lake_audit_2026_05_09_1217.md


/tmp/ipykernel_4891/1734131184.py:3: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  RUN_TAG = datetime.utcnow().strftime('%Y_%m_%d_%H%M')


In [ ]:
%cd $BACKEND_DIR
!python collabs/audit_data_lake.py \
  --drive-root "$DRIVE_ROOT" \
  --max-columns 25 \
  --sample-rows 2 \
  --output-md "$AUDIT_MD" \
  --output-json "$AUDIT_JSON"

!head -60 "$AUDIT_MD"


/content/pfein/back_end


^C
head: cannot open '/content/drive/MyDrive/PFE ML Data/pfe_data/reports/data_lake_audit_2026_05_09_1217.md' for reading: No such file or directory


## 7. BODACC Download Only

Use this on normal RAM. It downloads archives to Drive but does not export/process them.

In [ ]:
%cd $BACKEND_DIR
!python collabs/download_bodacc.py \
  --drive-root "$DRIVE_ROOT" \
  --repo-dir "$BACKEND_DIR" \
  --mode historical \
  --families PCL,RCS-B \
  --start-year 2017 \
  --end-year 2025 \
  --download \
  --no-export


## 8. BODACC Smoke Test

Optional. Use this if you want to test only a tiny BODACC download first.

In [ ]:
%cd $BACKEND_DIR
!python collabs/download_bodacc.py \
  --drive-root "$DRIVE_ROOT" \
  --repo-dir "$BACKEND_DIR" \
  --mode historical \
  --families PCL,RCS-B \
  --start-year 2025 \
  --end-year 2025 \
  --max-files 2 \
  --download \
  --no-export


## 9. BODACC Export + Rebuild

Use this after BODACC archives are downloaded. Enable High RAM if export/build hits memory pressure.

In [4]:
%cd $BACKEND_DIR
!python collabs/download_bodacc.py \
  --drive-root "$DRIVE_ROOT" \
  --repo-dir "$BACKEND_DIR" \
  --mode historical \
  --families PCL,RCS-B \
  --start-year 2017 \
  --end-year 2025 \
  --no-download \
  --export


/content/pfein/back_end
[run] /usr/bin/python3 -u -m app.tools.bodacc_archives_to_parquet --input-dir /content/drive/MyDrive/PFE ML Data/pfe_data/source-archives/bodacc --output-dir /content/drive/MyDrive/PFE ML Data/pfe_data/data-lake --progress-file /content/drive/MyDrive/PFE ML Data/pfe_data/data-lake/raw/bodacc/_batch_progress.json --mode historical --families PCL RCS-B --skip-existing
2026-05-09 12:19:55,538 | INFO | bodacc_archives_to_parquet | batch export discovered=2520 input_dir=/content/drive/MyDrive/PFE ML Data/pfe_data/source-archives/bodacc families=PCL,RCS-B
2026-05-09 12:19:56,400 | INFO | numexpr.utils | NumExpr defaulting to 8 threads.
2026-05-09 12:19:56,607 | INFO | bodacc_to_parquet | wrote part=00001 batch_rows=721 total_rows=721
2026-05-09 12:19:56,636 | INFO | bodacc_archives_to_parquet | exported archive=/content/drive/MyDrive/PFE ML Data/pfe_data/source-archives/bodacc/historical/2017/PCL/PCL_BXA20170001.taz rows=721
2026-05-09 12:19:57,176 | INFO | bodacc_to_

In [8]:
%cd $BACKEND_DIR
!python collabs/build_ml_data.py \
  --drive-root "$DRIVE_ROOT" \
  --start-year 2017 \
  --end-year 2025 \
  --max-companies 100000


/content/pfein/back_end
[run] /usr/bin/python3 -u -m app.tools.build_clean_core_sources --data-lake-dir /content/drive/MyDrive/PFE ML Data/pfe_data/data-lake --overwrite
2026-05-09 13:10:22,680 | INFO | build_clean_core_sources | clean core build started data_lake=/content/drive/MyDrive/PFE ML Data/pfe_data/data-lake overwrite=True
2026-05-09 13:10:22,734 | INFO | build_clean_core_sources | clean dataset=company_identity reading raw_root=/content/drive/MyDrive/PFE ML Data/pfe_data/data-lake/raw/insee/bulk/stock_unite_legale
2026-05-09 13:10:22,735 | INFO | build_clean_core_sources | clean dataset=company_identity writing output=/content/drive/MyDrive/PFE ML Data/pfe_data/data-lake/clean/company_identity/company_identity.parquet
100% ▕████████████████████████████████████████████████████████████▏ 
2026-05-09 13:11:11,443 | INFO | build_clean_core_sources | clean dataset=company_identity rows=29572772 output=/content/drive/MyDrive/PFE ML Data/pfe_data/data-lake/clean/company_identity
2026

## 10. INPI Credentials And Download

Only run this when you have INPI FTP/SFTP credentials. Do not commit credentials to Git.

In [9]:
import os

os.environ['INPI_FTP_HOST'] = 'www.inpi.net'
os.environ['INPI_FTP_PORT'] = '21'
os.environ['INPI_FTP_USER'] = 'rneinpiro'
os.environ['INPI_FTP_PASSWORD'] = 'vv8_rQ5f4M_2-E'
os.environ['INPI_FTP_PROTOCOL'] = 'ftp'
os.environ['INPI_REMOTE_BASE_DIR'] = '/'


In [ ]:
%cd $BACKEND_DIR
!python collabs/download_inpi.py \
  --drive-root "$DRIVE_ROOT" \
  --repo-dir "$BACKEND_DIR" \
  --categories comptes_annuels,formalites \
  --niveaux standard,niveau1 \
  --max-files 2


/content/pfein/back_end
[inpi] connected protocol=ftp host=www.inpi.net port=21
[inpi] discovered 4 matching archive(s)
[inpi] stock_RNE_comptes_annuels_20250926_1000_v2.zip: 100.0 MB / 3,493.4 MB (2.86%), 2.3 MB/s
[inpi] stock_RNE_comptes_annuels_20250926_1000_v2.zip: 200.0 MB / 3,493.4 MB (5.73%), 1.6 MB/s
[inpi] stock_RNE_comptes_annuels_20250926_1000_v2.zip: 300.0 MB / 3,493.4 MB (8.59%), 1.3 MB/s
[inpi] stock_RNE_comptes_annuels_20250926_1000_v2.zip: 400.0 MB / 3,493.4 MB (11.45%), 1.2 MB/s


## 11. INPI Export + Rebuild

After INPI ZIP archives are downloaded, export them to raw Parquet and rebuild features.

In [ ]:
%cd $BACKEND_DIR
!python collabs/export_raw_sources.py \
  --drive-root "$DRIVE_ROOT" \
  --repo-dir "$BACKEND_DIR" \
  --no-insee \
  --inpi \
  --no-bodacc


In [ ]:
%cd $BACKEND_DIR
!python collabs/build_ml_data.py \
  --drive-root "$DRIVE_ROOT" \
  --start-year 2017 \
  --end-year 2025 \
  --max-companies 100000


## 12. Training Gate

Do not train the final model until the audit shows enough source coverage and positive labels. For a capped baseline only:

In [ ]:
%cd $BACKEND_DIR
!python collabs/build_ml_data.py \
  --drive-root "$DRIVE_ROOT" \
  --start-year 2017 \
  --end-year 2025 \
  --max-companies 100000 \
  --train


## 13. Troubleshooting Cells

In [6]:
# Disk and folder sizes
!df -h /content
!du -h -d 1 /content 2>/dev/null | sort -h | tail -20


Filesystem      Size  Used Avail Use% Mounted on
overlay         226G   25G  202G  11% /
148K	/content/.config
3.0M	/content/pfein
55M	/content/sample_data
19G	/content
19G	/content/drive


In [ ]:
# Clear DuckDB temp only. Do not delete the Drive data folder.
!rm -rf /content/pfein_duckdb_tmp
!mkdir -p /content/pfein_duckdb_tmp
!ls -lah /content/pfein_duckdb_tmp


In [ ]:
# Show generated reports
!ls -lah "$DRIVE_ROOT/reports"
